In [ ]:
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

import json

In [2]:
llm = ChatOllama(
    model="gpt-oss:20b",
    temperature=0.3,
    num_predict=1000,
)

In [3]:
from langchain_core.tools import tool

from datetime import datetime, timedelta

@tool
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    """Returns the current date and time formatted according to the provided format string.
    Args:
        date_format (str): The format string for the output datetime, using Python's strftime format codes.
    Returns:
        str: The current date and time as a formatted string."""
    if not date_format:
        raise ValueError("date_format cannot be empty.")
    now = datetime.now()
    return now.strftime(date_format)

@tool
def add_duration_to_datetime(
    datetime_str:str, duration:int=0, unit:str="days", input_format:str="%Y-%m-%d"
):
    """
    Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.
    This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime.
    It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years.
    The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').
    
    Args:
        datetime_str (str): The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.
        duration (int): The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates).
        unit (str): The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'.
        input_format (str): The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'.
    Required:["datetime_str"].
    Returns:
        str: The resulting datetime after adding the specified duration, formatted in a detailed string format.
    Raises:
        ValueError: If the unit is not one of the supported time units."""

    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

@tool
def set_reminder(content: str, timestamp: str):
    """Creates a timed reminder that will notify the user at the specified time with the provided content.
    This tool schedules a notification to be delivered to the user at the exact timestamp provided.
    It should be used when a user wants to be reminded about something specific at a future point in time.
    The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives.
    
    Args:
        content (str): The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.
        timestamp (str): The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp.The system handles all timezone processing internally.
    Required: ["content", "timestamp"].
    Returns:
        None
    """
    msg = f"Reminder set for {timestamp}: {content}"
    print(f"----\n{msg}\n----")
    return msg

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [4]:
tools = [
    get_current_datetime,
    add_duration_to_datetime,
    set_reminder
    ]

tools_by_name = {tool.name: tool for tool in tools}
llm_with_tools = llm.bind_tools(tools)

In [5]:
def use_tools(ai_msg, tools_by_name, messages):

    for tool_call in ai_msg.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        tool_call_id = tool_call["id"]

        # print(type(tool_args), tool_args)

        if tool_name in tools_by_name:
            tool = tools_by_name[tool_name]
            try:
                result = tool.invoke(tool_args)
                print(f"Tool '{tool_name}'. Args: {tool_args}. Result: {result}")
                messages.append(ToolMessage( content=str(result),tool_call_id=tool_call_id,))
            except Exception as e:
                result = f"Error invoking tool '{tool_name}': {str(e)}"
                messages.append(ToolMessage( content=str(result),tool_call_id=tool_call_id,))
                print(f"Tool '{tool_name}'. Args: {tool_args}. Result: {result}")
        else:
            print(f"Tool '{tool_name}' not found.")
            return tools_by_name
    
    return

In [6]:
def chat(prompt, iterations=10):
    messages = [
        SystemMessage(content="You are a helpful assistant. Use tools when needed. You can call tools multiple times if needed."),
        HumanMessage(content=str(prompt))
    ]

    ai_msg = None
    
    for _ in range(iterations):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)
        print(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg

        use_tools(ai_msg, tools_by_name, messages)
    
    return ai_msg
    

In [ ]:
response = chat("What is the current time in HH:MM format?")
print(response.content)

In [ ]:
response = chat("What is the current time in HH:MM format? Also what is the current time in SS format?")
print(response.content)

In [ ]:
response = chat("The given date is 2026-07-06. What date is 103 days from the given date?")
print(response.content)

In [7]:
response = chat("Set a reminder when I should stop taking my medication. Its 42 days from today.")
print(response.content)

content='' additional_kwargs={} response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-07-07T11:56:30.305264Z', 'done': True, 'done_reason': 'stop', 'total_duration': 11610956458, 'load_duration': 271221125, 'prompt_eval_count': 816, 'prompt_eval_duration': 27810000, 'eval_count': 685, 'eval_duration': 11302917000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'} id='lc_run--019f3c6f-a947-76f1-b121-248afc9b4d3e-0' tool_calls=[{'name': 'set_reminder', 'args': {'content': 'Stop taking medication', 'timestamp': '2026-08-18T00:00:00'}, 'id': 'f304d15c-55f1-4048-ae41-3010be272253', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 816, 'output_tokens': 685, 'total_tokens': 1501}
----
Reminder set for 2026-08-18T00:00:00: Stop taking medication
----
Tool 'set_reminder'. Args: {'content': 'Stop taking medication', 'timestamp': '2026-08-18T00:00:00'}. Result: Reminder set for 2026-08-18T00:00:00: Stop taking medication
content='✅ Remi